# 🌫️ Spatio-Temporal Air Pollution Forecasting
## GNN + LSTM | USA EPA Dataset 2022–2023

**Expected runtime: ~45–60 min end-to-end**

### Files expected in `datasets/`
```
Gases:        hourly_44201_2022.zip  hourly_44201_2023.zip   (Ozone)
              hourly_42401_2022.zip  hourly_42401_2023.zip   (SO2)
              hourly_42101_2022.zip  hourly_42101_2023.zip   (CO)
              hourly_42602_2022.zip  hourly_42602_2023.zip   (NO2)
Particulates: hourly_88101_2022.zip  hourly_88101_2023.zip   (PM2.5 FRM)
              hourly_88502_2022.zip  hourly_88502_2023.zip   (PM2.5 non-FRM)
              hourly_81102_2022.zip  hourly_81102_2023.zip   (PM10)
              hourly_86101_2022.zip  hourly_86101_2023.zip   (PMc)
```

---
## 📦 Cell 1 — Imports & Configuration

In [1]:
# ── Install (run once, then comment out) ──────────────────────────────────────
# !pip install pyspark happybase scikit-learn pandas numpy matplotlib seaborn tqdm
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# !pip install torch_geometric

import os, warnings, zipfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from tqdm import tqdm
import pickle

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as TF
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# PyTorch Geometric
from torch_geometric.nn import GCNConv

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import BallTree

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'axes.titleweight': 'bold'})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Directory layout ──────────────────────────────────────────────────────────
DATASET_DIR = Path('datasets')           # ← your zip files live here
EXTRACT_DIR = Path('data/extracted');  EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED   = Path('data/processed');  PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES     = Path('data/figures');    FIGURES.mkdir(parents=True, exist_ok=True)
R_EXPORT    = Path('data/r_exports');  R_EXPORT.mkdir(parents=True, exist_ok=True)

# ── Exact file registry ───────────────────────────────────────────────────────
YEARS = [2022, 2023]

POLLUTANTS = {
    'pm25':  {'code': '88101', 'name': 'PM2.5 FRM',    'unit': 'µg/m³', 'std': 35.0,  'max_valid': 500.0},
    'pm25b': {'code': '88502', 'name': 'PM2.5 nonFRM', 'unit': 'µg/m³', 'std': 35.0,  'max_valid': 500.0},
    'pm10':  {'code': '81102', 'name': 'PM10',          'unit': 'µg/m³', 'std': 150.0, 'max_valid': 1000.0},
    'pmc':   {'code': '86101', 'name': 'PMc',           'unit': 'µg/m³', 'std': 150.0, 'max_valid': 500.0},
    'ozone': {'code': '44201', 'name': 'Ozone',         'unit': 'ppm',   'std': 0.07,  'max_valid': 0.5},
    'so2':   {'code': '42401', 'name': 'SO2',           'unit': 'ppb',   'std': 75.0,  'max_valid': 1000.0},
    'co':    {'code': '42101', 'name': 'CO',            'unit': 'ppm',   'std': 9.0,   'max_valid': 50.0},
    'no2':   {'code': '42602', 'name': 'NO2',           'unit': 'ppb',   'std': 100.0, 'max_valid': 500.0},
}

PRIMARY_POLLUTANT = 'pm25'   # GNN+LSTM target

print(f'✅ Device: {DEVICE}')
print(f'✅ Torch:  {torch.__version__}')
print('\n📁 Checking files in datasets/:')
all_ok = True
for p, meta in POLLUTANTS.items():
    for yr in YEARS:
        f = DATASET_DIR / f"hourly_{meta['code']}_{yr}.zip"
        ok = f.exists()
        if not ok: all_ok = False
        print(f"  {'✅' if ok else '❌ MISSING'}  {f.name}")
print(f'\n{"✅ All files found!" if all_ok else "⚠️  Some files missing — check paths above"}')

✅ Device: cpu
✅ Torch:  2.8.0

📁 Checking files in datasets/:
  ✅  hourly_88101_2022.zip
  ✅  hourly_88101_2023.zip
  ✅  hourly_88502_2022.zip
  ✅  hourly_88502_2023.zip
  ✅  hourly_81102_2022.zip
  ✅  hourly_81102_2023.zip
  ✅  hourly_86101_2022.zip
  ✅  hourly_86101_2023.zip
  ✅  hourly_44201_2022.zip
  ✅  hourly_44201_2023.zip
  ✅  hourly_42401_2022.zip
  ✅  hourly_42401_2023.zip
  ✅  hourly_42101_2022.zip
  ✅  hourly_42101_2023.zip
  ✅  hourly_42602_2022.zip
  ✅  hourly_42602_2023.zip

✅ All files found!


---
## ⚡ Cell 2 — PySpark: Extract ZIPs + Clean All Pollutants

In [2]:
spark = (
    SparkSession.builder
    .appName('EPA_AirPollution_Forecasting')
    .config('spark.driver.memory', '10g')
    .config('spark.executor.memory', '10g')
    .config('spark.sql.shuffle.partitions', '100')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'✅ Spark {spark.version}  →  UI at http://localhost:4040')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 23:17:21 WARN Utils: Your hostname, Mitens-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.0.0.2 instead (on interface en0)
26/04/21 23:17:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 23:17:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 4.1.1  →  UI at http://localhost:4040


In [3]:
# ── Step 1: Extract all ZIPs ──────────────────────────────────────────────────
def extract_zip(zip_path: Path) -> Path:
    stem    = zip_path.stem                          # e.g. hourly_88101_2022
    out_csv = EXTRACT_DIR / f'{stem}.csv'
    if out_csv.exists():
        return out_csv
    print(f'  📦 Extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        csv_members = [n for n in z.namelist() if n.lower().endswith('.csv')]
        z.extract(csv_members[0], EXTRACT_DIR)
        (EXTRACT_DIR / csv_members[0]).rename(out_csv)
    return out_csv

extracted_files = {}   # pollutant_key → [csv_path, ...]
for p, meta in POLLUTANTS.items():
    paths = []
    for yr in YEARS:
        zp = DATASET_DIR / f"hourly_{meta['code']}_{yr}.zip"
        if zp.exists():
            paths.append(str(extract_zip(zp)))
    extracted_files[p] = paths
    print(f'  {p:6s} → {len(paths)} file(s)')

print('\n✅ Extraction complete.')

  pm25   → 2 file(s)
  pm25b  → 2 file(s)
  pm10   → 2 file(s)
  pmc    → 2 file(s)
  ozone  → 2 file(s)
  so2    → 2 file(s)
  co     → 2 file(s)
  no2    → 2 file(s)

✅ Extraction complete.


In [4]:
# ── Step 2: Spark schema + loader ─────────────────────────────────────────────
# NOTE: EPA column names use underscores — we rename to our standard snake_case

def load_pollutant_spark(csv_paths, pollutant_key, max_valid):
    """
    Loads multi-year EPA CSV files for one pollutant.
    Returns cleaned Spark DataFrame:
      station_id | timestamp | latitude | longitude | state_name | county_name | value | pollutant
    """
    df = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .option('multiLine', 'false')
        .csv(csv_paths)
    )

    # Build rename map: key = normalised column name (stripped, lowercased, underscores),
    # value = our target name.
    # EPA files use underscores consistently (e.g. "State_Code", "Sample_Measurement").
    rename_map = {
        'state_code':         'state_code',
        'county_code':        'county_code',
        'site_num':           'site_num',
        'latitude':           'latitude',
        'longitude':          'longitude',
        'date_local':         'date_local',
        'time_local':         'time_local',
        'sample_measurement': 'value',      # EPA col = "Sample_Measurement"
        'state_name':         'state_name',
        'county_name':        'county_name',
    }

    # Resilient rename: normalise each actual column → check map → rename if found
    for col in df.columns:
        normalised = col.strip().lower().replace(' ', '_')
        if normalised in rename_map:
            target = rename_map[normalised]
            if col != target:                # avoid no-op rename
                df = df.withColumnRenamed(col, target)

    # Sanity-check that the measurement column was found
    if 'value' not in df.columns:
        raise ValueError(
            f"❌ Failed to locate 'Sample_Measurement' for pollutant '{pollutant_key}'.\n"
            f"   Actual columns: {df.columns}"
        )

    df = (
        df
        # Unique station ID
        .withColumn('station_id', F.concat_ws('-',
            F.col('state_code'), F.col('county_code'), F.col('site_num')))
        # Unified timestamp
        .withColumn('timestamp',
            F.to_timestamp(F.concat_ws(' ', F.col('date_local'), F.col('time_local')),
                           'yyyy-MM-dd HH:mm'))
        # Quality filters
        .filter(F.col('value').isNotNull())
        .filter(F.col('value') >= 0)
        .filter(F.col('value') <= max_valid)
        .filter(F.col('timestamp').isNotNull())
        .filter(F.col('latitude').isNotNull())
        # Add pollutant label
        .withColumn('pollutant', F.lit(pollutant_key))
        # Select final columns
        .select('station_id', 'timestamp', 'latitude', 'longitude',
                'state_name', 'county_name', 'value', 'pollutant')
        .dropDuplicates(['station_id', 'timestamp', 'pollutant'])
    )
    return df


# ── Step 3: Load all pollutants ────────────────────────────────────────────────
print('⏳ Loading all pollutants into Spark (this takes ~5-8 min)...')
t0 = time.time()

spark_dfs = {}
row_counts = {}

for p, meta in POLLUTANTS.items():
    if not extracted_files[p]:
        continue
    df = load_pollutant_spark(extracted_files[p], p, meta['max_valid'])
    df.cache()
    cnt = df.count()
    spark_dfs[p]   = df
    row_counts[p]  = cnt
    print(f'  {meta["name"]:20s}  {cnt:>12,} rows')

print(f'\n✅ Done in {(time.time()-t0)/60:.1f} min')
print(f'   Total rows across all pollutants: {sum(row_counts.values()):,}')


⏳ Loading all pollutants into Spark (this takes ~5-8 min)...


{"ts": "2026-04-21 23:00:13.697", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `value` cannot be resolved. Did you mean one of the following? [`Datum`, `MDL`, `POC`, `latitude`, `Qualifier`]. SQLSTATE: 42703", "context": {"file": "line 45 in cell [4]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o72.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `value` cannot be resolved. Did you mean one of the following? [`Datum`, `MDL`, `POC`, `latitude`, `Qualifier`]. SQLSTATE: 42703;\n'Filter 'isNotNull('value)\n+- Project [State_Code#17, County_Code#18, Site_Num#19, Parameter_Code#20, POC#21, latitude#42, longitude#43, Datum#24, Parameter_Name#25, Date_Local#26, Time_Local#27, Dat

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `value` cannot be resolved. Did you mean one of the following? [`Datum`, `MDL`, `POC`, `latitude`, `Qualifier`]. SQLSTATE: 42703;
'Filter 'isNotNull('value)
+- Project [State_Code#17, County_Code#18, Site_Num#19, Parameter_Code#20, POC#21, latitude#42, longitude#43, Datum#24, Parameter_Name#25, Date_Local#26, Time_Local#27, Date_GMT#28, Time_GMT#29, Sample_Measurement#30, Units_of_Measure#31, MDL#32, Uncertainty#33, Qualifier#34, Method_Type#35, Method_Code#36, Method_Name#37, State_Name#38, County_Name#39, Date_of_Last_Change#40, station_id#44, ... 1 more fields]
   +- Project [State_Code#17, County_Code#18, Site_Num#19, Parameter_Code#20, POC#21, latitude#42, longitude#43, Datum#24, Parameter_Name#25, Date_Local#26, Time_Local#27, Date_GMT#28, Time_GMT#29, Sample_Measurement#30, Units_of_Measure#31, MDL#32, Uncertainty#33, Qualifier#34, Method_Type#35, Method_Code#36, Method_Nam

In [12]:
# (duplicate cell removed — see cell above)


⏳ Loading all pollutants into Spark (this takes ~5-8 min)...


ValueError: ❌ Failed to find/rename 'Sample Measurement' in pm25! Actual columns: ['State_Code', 'County_Code', 'Site_Num', 'Parameter_Code', 'POC', 'latitude', 'longitude', 'Datum', 'Parameter_Name', 'Date_Local', 'Time_Local', 'Date_GMT', 'Time_GMT', 'Sample_Measurement', 'Units_of_Measure', 'MDL', 'Uncertainty', 'Qualifier', 'Method_Type', 'Method_Code', 'Method_Name', 'State_Name', 'County_Name', 'Date_of_Last_Change']

In [13]:
# ── Step 4: Feature engineering on primary pollutant (PM2.5) ──────────────────
def engineer_features(df):
    """
    Adds temporal features, lag features, and rolling statistics.
    All computed distributed via PySpark Window functions.
    """
    t_window = Window.partitionBy('station_id').orderBy(F.col('timestamp').cast('long'))
    r_window = t_window.rowsBetween(-23, 0)  # 24-hour rolling

    df = (
        df
        # Temporal encoding
        .withColumn('hour',        F.hour('timestamp'))
        .withColumn('day_of_week', F.dayofweek('timestamp'))
        .withColumn('month',       F.month('timestamp'))
        .withColumn('year',        F.year('timestamp'))
        .withColumn('is_weekend',  (F.dayofweek('timestamp').isin([1,7])).cast(IntegerType()))
        # Cyclical encoding
        .withColumn('hour_sin',    F.sin(2 * 3.14159 * F.col('hour') / 24))
        .withColumn('hour_cos',    F.cos(2 * 3.14159 * F.col('hour') / 24))
        .withColumn('month_sin',   F.sin(2 * 3.14159 * F.col('month') / 12))
        .withColumn('month_cos',   F.cos(2 * 3.14159 * F.col('month') / 12))
        .withColumn('dow_sin',     F.sin(2 * 3.14159 * F.col('day_of_week') / 7))
        .withColumn('dow_cos',     F.cos(2 * 3.14159 * F.col('day_of_week') / 7))
        # Lag features
        .withColumn('lag_1h',  F.lag('value', 1).over(t_window))
        .withColumn('lag_3h',  F.lag('value', 3).over(t_window))
        .withColumn('lag_6h',  F.lag('value', 6).over(t_window))
        .withColumn('lag_12h', F.lag('value', 12).over(t_window))
        .withColumn('lag_24h', F.lag('value', 24).over(t_window))
        # Rolling statistics
        .withColumn('roll_24h_mean', F.mean('value').over(r_window))
        .withColumn('roll_24h_std',  F.stddev('value').over(r_window))
        .withColumn('roll_24h_max',  F.max('value').over(r_window))
        # Drop rows with null lags (first 24 rows per station)
        .dropna(subset=['lag_24h'])
    )
    return df


print('⏳ Engineering features on PM2.5...')
t0 = time.time()
pm25_spark = engineer_features(spark_dfs[PRIMARY_POLLUTANT])
pm25_spark.cache()
n_pm25 = pm25_spark.count()
print(f'✅ PM2.5 after feature engineering: {n_pm25:,} rows  ({(time.time()-t0)/60:.1f} min)')
pm25_spark.printSchema()

⏳ Engineering features on PM2.5...


KeyError: 'pm25'

In [ ]:
# ── Step 5: Filter well-covered stations & save Parquet ───────────────────────
MIN_READINGS = 6000   # ~250 days of hourly coverage

station_counts_spark = (
    pm25_spark
    .groupBy('station_id', 'latitude', 'longitude', 'state_name')
    .agg(F.count('*').alias('n_readings'),
         F.mean('value').alias('mean_pm25'),
         F.stddev('value').alias('std_pm25'))
    .filter(F.col('n_readings') >= MIN_READINGS)
)

pm25_filtered = pm25_spark.join(
    station_counts_spark.select('station_id'), on='station_id', how='inner'
)

N_STATIONS = station_counts_spark.count()
print(f'✅ Stations retained (≥{MIN_READINGS} readings): {N_STATIONS}')

# Save as Parquet — columnar, compressed, ~5x faster to reload than CSV
PARQUET_PATH = str(PROCESSED / 'pm25_engineered.parquet')
pm25_filtered.write.mode('overwrite').parquet(PARQUET_PATH)
print(f'✅ Parquet saved → {PARQUET_PATH}')

# Station metadata to CSV
station_meta_pd = station_counts_spark.toPandas()
station_meta_pd.to_csv(PROCESSED / 'station_metadata.csv', index=False)
print(f'✅ Station metadata: {len(station_meta_pd)} stations saved')

In [ ]:
# ── Step 6: Spark Summary Statistics (Big Data analytics) ─────────────────────
print('='*60)
print('  📊 Spark Distributed Summary Statistics')
print('='*60)

print('\n🔹 PM2.5 Distribution:')
pm25_filtered.select('value').describe().show()

print('\n🔹 Top 10 States by Station Count:')
(
    pm25_filtered
    .groupBy('state_name')
    .agg(
        F.countDistinct('station_id').alias('stations'),
        F.round(F.mean('value'), 3).alias('mean_pm25'),
        F.round(F.max('value'), 1).alias('max_pm25'),
        F.count('*').alias('total_rows')
    )
    .orderBy(F.desc('stations'))
    .show(10, truncate=False)
)

print('\n🔹 Hourly average PM2.5 (Diurnal Pattern):')
(
    pm25_filtered
    .groupBy('hour')
    .agg(F.round(F.mean('value'), 3).alias('avg_pm25'))
    .orderBy('hour')
    .show(24)
)

print('\n🔹 Row counts by pollutant (all loaded):')
for p, cnt in row_counts.items():
    bar = '█' * int(cnt / max(row_counts.values()) * 30)
    print(f'  {POLLUTANTS[p]["name"]:22s} {cnt:>12,}  {bar}')

---
## 🗄️ Cell 3 — HBase Storage via HappyBase

In [ ]:
# ── HBase: requires Docker running ────────────────────────────────────────────
# Start HBase:  docker run -d -p 2181:2181 -p 9090:9090 harisekhon/hbase
# Install:      pip install happybase

import happybase

HBASE_HOST  = 'localhost'
HBASE_TABLE = 'epa_air_quality'

def hbase_setup(host, table_name):
    conn = happybase.Connection(host)
    existing = [t.decode() for t in conn.tables()]
    if table_name not in existing:
        conn.create_table(table_name, {
            'cf_raw':      dict(max_versions=1),  # raw measurements
            'cf_features': dict(max_versions=1),  # engineered features
            'cf_meta':     dict(max_versions=1),  # station metadata
        })
        print(f'✅ HBase table "{table_name}" created with 3 column families.')
    else:
        print(f'ℹ️  Table "{table_name}" already exists.')
    conn.close()


def hbase_batch_write(pandas_df: pd.DataFrame, host, table_name, batch_size=2000):
    """
    Writes DataFrame rows to HBase.
    Row key design:  station_id#YYYYMMDD#HH
    e.g.  '06-037-1103#20230601#14'
    """
    conn  = happybase.Connection(host)
    table = conn.table(table_name)
    written = 0
    with table.batch(batch_size=batch_size) as b:
        for _, row in tqdm(pandas_df.iterrows(), total=len(pandas_df),
                           desc='HBase write'):
            ts  = pd.Timestamp(row['timestamp'])
            key = f"{row['station_id']}#{ts.strftime('%Y%m%d')}#{ts.strftime('%H')}"
            b.put(key.encode(), {
                b'cf_raw:pm25':          str(row['value']).encode(),
                b'cf_raw:year':          str(row.get('year','')).encode(),
                b'cf_features:lag_1h':   str(row.get('lag_1h','')).encode(),
                b'cf_features:lag_24h':  str(row.get('lag_24h','')).encode(),
                b'cf_features:roll_mean':str(row.get('roll_24h_mean','')).encode(),
                b'cf_features:hour_sin': str(row.get('hour_sin','')).encode(),
                b'cf_features:hour_cos': str(row.get('hour_cos','')).encode(),
                b'cf_meta:latitude':     str(row['latitude']).encode(),
                b'cf_meta:longitude':    str(row['longitude']).encode(),
                b'cf_meta:state':        str(row.get('state_name','')).encode(),
            })
            written += 1
    conn.close()
    print(f'✅ Written {written:,} rows to HBase.')


def hbase_read_station(station_id, date_from, date_to, host, table_name):
    """
    Range-scan for one station over a date window.
    date_from/date_to: 'YYYYMMDD'
    """
    conn   = happybase.Connection(host)
    table  = conn.table(table_name)
    start  = f'{station_id}#{date_from}#00'.encode()
    stop   = f'{station_id}#{date_to}#23'.encode()
    rows   = []
    for key, data in table.scan(row_start=start, row_stop=stop):
        rows.append({
            'row_key':   key.decode(),
            'pm25':      float(data.get(b'cf_raw:pm25', b'nan')),
            'lag_1h':    float(data.get(b'cf_features:lag_1h', b'nan')),
            'roll_mean': float(data.get(b'cf_features:roll_mean', b'nan')),
            'latitude':  float(data.get(b'cf_meta:latitude', b'0')),
            'longitude': float(data.get(b'cf_meta:longitude', b'0')),
        })
    conn.close()
    return pd.DataFrame(rows)


# ── Write sample (first 10k rows) to HBase ────────────────────────────────────
try:
    sample_pdf = pm25_filtered.limit(10000).toPandas()
    hbase_setup(HBASE_HOST, HBASE_TABLE)
    hbase_batch_write(sample_pdf, HBASE_HOST, HBASE_TABLE)
    print('\n📖 Reading back sample station from HBase...')
    sample_station = sample_pdf['station_id'].iloc[0]
    result = hbase_read_station(sample_station, '20220101', '20221231',
                                 HBASE_HOST, HBASE_TABLE)
    print(result.head())
except Exception as e:
    print(f'⚠️  HBase unavailable: {e}')
    print('   → Start with: docker run -d -p 2181:2181 -p 9090:9090 harisekhon/hbase')
    print('   → Pipeline continues using Parquet as storage backend.')

---
## 📊 Cell 4 — EDA Visualizations (Python/Matplotlib)

In [ ]:
# ── Reload from Parquet for pandas-level EDA ─────────────────────────────────
pm25_pd = spark.read.parquet(PARQUET_PATH).toPandas()
pm25_pd['timestamp'] = pd.to_datetime(pm25_pd['timestamp'])
station_meta = pd.read_csv(PROCESSED / 'station_metadata.csv')

print(f'Loaded: {pm25_pd.shape[0]:,} rows  |  {pm25_pd["station_id"].nunique()} stations')

In [ ]:
# ── Figure 1: Multi-Pollutant Row-Count Comparison (Big Data scale) ───────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart — row counts
names  = [POLLUTANTS[p]['name'] for p in row_counts]
counts = [row_counts[p]/1e6 for p in row_counts]
colors = sns.color_palette('tab10', len(counts))

bars = axes[0].barh(names, counts, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_xlabel('Rows (millions)', fontsize=12)
axes[0].set_title('Dataset Scale per Pollutant\n(2022–2023 combined)', fontsize=13)
for bar, val in zip(bars, counts):
    axes[0].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}M', va='center', fontsize=10)
axes[0].set_xlim(0, max(counts)*1.18)

# Pie chart — proportion
axes[1].pie(counts, labels=names, colors=colors, autopct='%1.1f%%',
            startangle=140, pctdistance=0.82,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Row Distribution Across Pollutants', fontsize=13)

plt.suptitle('EPA AQS Dataset — Big Data Scale Overview', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'fig1_dataset_scale.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 1 saved.')

In [ ]:
# ── Figure 2: PM2.5 Distribution & Box Plots ──────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.3)

# 2a: Histogram with KDE
ax1 = fig.add_subplot(gs[0, 0])
sns.histplot(pm25_pd['value'].clip(0, 100), bins=80, kde=True,
             ax=ax1, color='steelblue', alpha=0.7)
ax1.axvline(35, color='red', ls='--', lw=1.5, label='EPA 24h std')
ax1.set_xlabel('PM2.5 (µg/m³)'); ax1.set_title('PM2.5 Distribution')
ax1.legend(fontsize=9)

# 2b: Log-scale histogram
ax2 = fig.add_subplot(gs[0, 1])
sns.histplot(pm25_pd['value'].clip(0, 200), bins=80, ax=ax2,
             color='teal', alpha=0.7, log_scale=(False, True))
ax2.axvline(35, color='red', ls='--', lw=1.5)
ax2.set_xlabel('PM2.5 (µg/m³)'); ax2.set_title('PM2.5 Distribution (Log Y)')

# 2c: Year comparison violin
ax3 = fig.add_subplot(gs[0, 2])
sample = pm25_pd.sample(min(50000, len(pm25_pd)), random_state=SEED)
sns.violinplot(data=sample, x='year', y='value', palette='Set2',
               inner='quartile', ax=ax3, cut=0)
ax3.set_ylim(0, 80); ax3.axhline(35, color='red', ls='--', lw=1.5)
ax3.set_xlabel('Year'); ax3.set_ylabel('PM2.5 (µg/m³)')
ax3.set_title('Year-over-Year Comparison')

# 2d: Monthly box plot
ax4 = fig.add_subplot(gs[1, :])
month_sample = pm25_pd.sample(min(100000, len(pm25_pd)), random_state=SEED)
month_names  = ['Jan','Feb','Mar','Apr','May','Jun',
                 'Jul','Aug','Sep','Oct','Nov','Dec']
sns.boxplot(data=month_sample, x='month', y='value',
            palette='coolwarm', showfliers=False, ax=ax4,
            order=range(1,13))
ax4.set_xticklabels(month_names)
ax4.axhline(35, color='red', ls='--', lw=1.5, label='EPA 24h Standard (35 µg/m³)')
ax4.set_xlabel('Month'); ax4.set_ylabel('PM2.5 (µg/m³)')
ax4.set_title('Monthly PM2.5 Distribution — Seasonal Patterns')
ax4.legend(fontsize=10)
ax4.set_ylim(0, 60)

plt.suptitle('PM2.5 Exploratory Data Analysis — 2022–2023', fontsize=16, y=1.01)
plt.savefig(FIGURES / 'fig2_pm25_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 2 saved.')

In [ ]:
# ── Figure 3: Diurnal & Weekly Heatmap ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 3a: Diurnal pattern (hour vs month heatmap)
heatmap_data = (
    pm25_pd.groupby(['month', 'hour'])['value']
    .mean().unstack(level='hour')
)
sns.heatmap(heatmap_data, ax=axes[0], cmap='YlOrRd',
            linewidths=0, annot=False, fmt='.1f',
            cbar_kws={'label': 'Mean PM2.5 (µg/m³)'})
axes[0].set_yticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
axes[0].set_xlabel('Hour of Day'); axes[0].set_ylabel('Month')
axes[0].set_title('PM2.5 Heatmap — Hour × Month\n(Diurnal + Seasonal Patterns)')

# 3b: Day-of-week × hour heatmap
heatmap_dow = (
    pm25_pd.groupby(['day_of_week', 'hour'])['value']
    .mean().unstack(level='hour')
)
sns.heatmap(heatmap_dow, ax=axes[1], cmap='YlOrRd',
            linewidths=0, cbar_kws={'label': 'Mean PM2.5 (µg/m³)'})
axes[1].set_yticklabels(['Sun','Mon','Tue','Wed','Thu','Fri','Sat'], rotation=0)
axes[1].set_xlabel('Hour of Day'); axes[1].set_ylabel('Day of Week')
axes[1].set_title('PM2.5 Heatmap — Day × Hour\n(Weekly Traffic Patterns)')

plt.suptitle('Temporal PM2.5 Patterns', fontsize=15)
plt.tight_layout()
plt.savefig(FIGURES / 'fig3_temporal_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 3 saved.')

In [ ]:
# ── Figure 4: Geographic Distribution Map ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 4a: Station locations colored by mean PM2.5
sc1 = axes[0].scatter(
    station_meta['longitude'], station_meta['latitude'],
    c=station_meta['mean_pm25'], cmap='RdYlGn_r',
    s=30, alpha=0.75, edgecolors='white', linewidths=0.2,
    vmin=0, vmax=20
)
plt.colorbar(sc1, ax=axes[0], label='Mean PM2.5 (µg/m³)', shrink=0.85)
axes[0].set_xlim(-130, -65); axes[0].set_ylim(24, 50)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_title(f'EPA Monitoring Stations\n({len(station_meta)} stations, mean PM2.5)', fontsize=13)
axes[0].set_aspect('equal')

# 4b: Number of readings per station
sc2 = axes[1].scatter(
    station_meta['longitude'], station_meta['latitude'],
    c=station_meta['n_readings'], cmap='Blues',
    s=30, alpha=0.8, edgecolors='white', linewidths=0.2
)
plt.colorbar(sc2, ax=axes[1], label='Number of hourly readings', shrink=0.85)
axes[1].set_xlim(-130, -65); axes[1].set_ylim(24, 50)
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].set_title('Data Coverage per Station\n(Number of readings)', fontsize=13)
axes[1].set_aspect('equal')

plt.suptitle('Spatial Distribution of EPA PM2.5 Monitoring Network — USA', fontsize=15)
plt.tight_layout()
plt.savefig(FIGURES / 'fig4_station_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 4 saved.')

In [ ]:
# ── Figure 5: State-Level Analysis ────────────────────────────────────────────
state_stats = (
    pm25_pd.groupby('state_name')['value']
    .agg(['mean', 'median', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean':'mean_pm25','median':'med_pm25','std':'std_pm25','count':'n_rows'})
    .sort_values('mean_pm25', ascending=True)
)
# EPA exceedance rate per state
exceed = (
    pm25_pd.groupby('state_name')
    .apply(lambda x: (x['value'] > 35).mean() * 100)
    .reset_index(name='exceed_pct')
)
state_stats = state_stats.merge(exceed, on='state_name')

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# 5a: Mean PM2.5 by state (horizontal bar)
top30 = state_stats.tail(30)
bars  = axes[0].barh(top30['state_name'], top30['mean_pm25'],
                     xerr=top30['std_pm25'], color=[
    '#d62728' if v > 12 else '#ff7f0e' if v > 8 else '#2ca02c'
    for v in top30['mean_pm25']], edgecolor='white', linewidth=0.6,
    error_kw=dict(elinewidth=0.8, ecolor='grey', capsize=2))
axes[0].axvline(12, color='orange', ls='--', lw=1.5, label='Annual NAAQS (12 µg/m³)')
axes[0].axvline(9,  color='green',  ls='--', lw=1.2, label='WHO Guideline (9 µg/m³)')
axes[0].set_xlabel('Mean PM2.5 (µg/m³)'); axes[0].set_ylabel('State')
axes[0].set_title('Top 30 States — Mean PM2.5\n(±1 std dev)', fontsize=13)
axes[0].legend(fontsize=9)

# 5b: EPA exceedance rate by state
top30_e = state_stats.sort_values('exceed_pct', ascending=True).tail(30)
colors_e = ['#d62728' if v > 5 else '#ff7f0e' if v > 2 else '#2ca02c'
            for v in top30_e['exceed_pct']]
axes[1].barh(top30_e['state_name'], top30_e['exceed_pct'],
             color=colors_e, edgecolor='white', linewidth=0.6)
axes[1].set_xlabel('Hours exceeding 35 µg/m³ (%)')
axes[1].set_ylabel('State')
axes[1].set_title('EPA 24h Standard Exceedance Rate\n(% of hourly readings > 35 µg/m³)', fontsize=13)

plt.suptitle('State-Level PM2.5 Analysis — 2022–2023', fontsize=15)
plt.tight_layout()
plt.savefig(FIGURES / 'fig5_state_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 5 saved.')

In [ ]:
# ── Figure 6: Lag Feature Correlation + Autocorrelation ───────────────────────
# Sample one well-covered station
top_station = station_meta.sort_values('n_readings', ascending=False).iloc[0]['station_id']
st_df = pm25_pd[pm25_pd['station_id'] == top_station].sort_values('timestamp')
print(f'Selected station: {top_station}  ({len(st_df)} readings)')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 6a: Time series (2 months)
ax = axes[0, 0]
ts2m = st_df.head(1440)  # 60 days
ax.plot(ts2m['timestamp'], ts2m['value'], color='steelblue', lw=0.8, alpha=0.9)
ax.axhline(35, color='red', ls='--', lw=1.2, label='EPA Std')
ax.fill_between(ts2m['timestamp'], 0, ts2m['value'],
                where=ts2m['value'] > 35, alpha=0.3, color='red')
ax.set_xlabel('Date'); ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('PM2.5 Time Series — 60 Day Window'); ax.legend()

# 6b: Lag correlation scatter (lag=1h)
lag_df = st_df.dropna(subset=['lag_1h', 'lag_24h'])
ax = axes[0, 1]
ax.scatter(lag_df['lag_1h'], lag_df['value'],
           alpha=0.2, s=8, color='teal')
r1 = np.corrcoef(lag_df['lag_1h'], lag_df['value'])[0,1]
ax.set_xlabel('PM2.5 at t-1h'); ax.set_ylabel('PM2.5 at t')
ax.set_title(f'Lag-1h Autocorrelation  (r = {r1:.3f})')

# 6c: Lag-24h scatter
ax = axes[1, 0]
ax.scatter(lag_df['lag_24h'], lag_df['value'],
           alpha=0.2, s=8, color='darkorange')
r24 = np.corrcoef(lag_df['lag_24h'], lag_df['value'])[0,1]
ax.set_xlabel('PM2.5 at t-24h'); ax.set_ylabel('PM2.5 at t')
ax.set_title(f'Lag-24h Autocorrelation  (r = {r24:.3f})')

# 6d: Autocorrelation bar chart across lags
ax = axes[1, 1]
lag_cols  = ['lag_1h', 'lag_3h', 'lag_6h', 'lag_12h', 'lag_24h']
lag_labels = ['1h', '3h', '6h', '12h', '24h']
lag_corrs  = [np.corrcoef(lag_df[c], lag_df['value'])[0,1]
              for c in lag_cols if c in lag_df.columns]
bar_colors = ['#2196F3' if c > 0 else '#F44336' for c in lag_corrs]
bars = ax.bar(lag_labels[:len(lag_corrs)], lag_corrs,
              color=bar_colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Lag'); ax.set_ylabel('Pearson Correlation')
ax.set_title('Autocorrelation by Lag — Supports LSTM design')
ax.set_ylim(-0.1, 1.0)
for bar, val in zip(bars, lag_corrs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', fontsize=9)

plt.suptitle(f'Autocorrelation Analysis — Station {top_station}', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'fig6_autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 6 saved.')

---
## 🕸️ Cell 5 — Graph Construction

In [ ]:
# ── Pivot: [timestamp × station] matrix ──────────────────────────────────────
pivot = (
    pm25_pd
    .pivot_table(index='timestamp', columns='station_id',
                 values='value', aggfunc='mean')
)

# Keep stations present in both years (better coverage)
thresh = int(0.75 * len(pivot))
pivot  = pivot.dropna(axis=1, thresh=thresh)
pivot  = pivot.ffill(limit=6).bfill(limit=6).fillna(pivot.median())

STATIONS = pivot.columns.tolist()
N        = len(STATIONS)
print(f'✅ Pivot matrix: {pivot.shape}  →  {N} stations × {len(pivot)} timesteps')

# Station coordinates aligned with pivot
station_loc = (
    pm25_pd.drop_duplicates('station_id')
    .set_index('station_id')[['latitude','longitude']]
    .reindex(STATIONS)
)
station_loc.dropna(inplace=True)
STATIONS = station_loc.index.tolist()
pivot    = pivot[STATIONS]
N        = len(STATIONS)
print(f'After coordinate alignment: {N} stations')

In [ ]:
# ── Build spatial graph with BallTree (haversine) ─────────────────────────────
RADIUS_KM = 150.0

coords_rad = np.radians(station_loc[['latitude','longitude']].values)
tree       = BallTree(coords_rad, metric='haversine')
r_rad      = RADIUS_KM / 6371.0

src, dst, weights_raw = [], [], []
for i, coord in enumerate(coords_rad):
    idxs, dists = tree.query_radius(coord.reshape(1,-1), r=r_rad, return_distance=True)
    for j, d in zip(idxs[0], dists[0]):
        if i != j:
            src.append(i); dst.append(j)
            weights_raw.append(1.0 / (d * 6371.0 + 1e-6))

edge_index  = torch.tensor([src, dst], dtype=torch.long)
edge_weight = torch.tensor(weights_raw, dtype=torch.float)
edge_weight = edge_weight / edge_weight.max()   # normalize to [0,1]

print(f'✅ Graph: {N} nodes | {edge_index.shape[1]:,} edges | '
      f'avg degree {edge_index.shape[1]/N:.1f}')

In [ ]:
# ── Figure 7: Graph Structure Visualization ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

lats = station_loc['latitude'].values
lons = station_loc['longitude'].values
degs = np.bincount(src, minlength=N)

# 7a: Station map with edges
ax = axes[0]
sample_edges = list(zip(src, dst))[:800]
for (i, j) in sample_edges:
    ax.plot([lons[i], lons[j]], [lats[i], lats[j]],
            'steelblue', alpha=0.08, lw=0.5)
sc = ax.scatter(lons, lats, c=degs, cmap='plasma',
                s=20, alpha=0.9, zorder=5)
plt.colorbar(sc, ax=ax, label='Node Degree')
ax.set_xlim(-130, -65); ax.set_ylim(24, 50)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Graph Structure\n({N} nodes, {len(src):,} edges, r={RADIUS_KM}km)')

# 7b: Degree distribution
ax = axes[1]
ax.hist(degs, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(degs.mean(), color='red', ls='--', lw=2, label=f'Mean={degs.mean():.1f}')
ax.set_xlabel('Node Degree (# spatial neighbors)')
ax.set_ylabel('Count')
ax.set_title('Degree Distribution\n(Validates graph connectivity)')
ax.legend()

# 7c: Edge weight distribution
ax = axes[2]
ew = edge_weight.numpy()
ax.hist(ew, bins=50, color='darkorange', edgecolor='white', alpha=0.85)
ax.axvline(ew.mean(), color='red', ls='--', lw=2, label=f'Mean={ew.mean():.3f}')
ax.set_xlabel('Edge Weight (inverse distance, normalized)')
ax.set_ylabel('Count')
ax.set_title('Edge Weight Distribution\n(Closer stations → higher weight)')
ax.legend()

plt.suptitle('Spatial Graph Analysis — GNN Input Structure', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'fig7_graph_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 7 saved.')

---
## 🧠 Cell 6 — GNN + LSTM Model

In [ ]:
# ── Normalize & create dataset ────────────────────────────────────────────────
T_IN  = 12   # past 12 hours as input
T_OUT = 1    # predict 1 hour ahead

scaler        = MinMaxScaler()
values_scaled = scaler.fit_transform(pivot.values)   # [T, N]

T_total = len(values_scaled)
t_train = int(0.70 * T_total)
t_val   = int(0.85 * T_total)

class SpatioTemporalDS(Dataset):
    def __init__(self, data, t_in, t_out):
        self.data  = torch.tensor(data, dtype=torch.float32)
        self.t_in  = t_in; self.t_out = t_out
    def __len__(self):
        return len(self.data) - self.t_in - self.t_out + 1
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.t_in]                           # [T_IN, N]
        y = self.data[idx+self.t_in:idx+self.t_in+self.t_out].squeeze(0)  # [N]
        return x, y

train_dl = DataLoader(SpatioTemporalDS(values_scaled[:t_train],   T_IN, T_OUT), batch_size=32, shuffle=True)
val_dl   = DataLoader(SpatioTemporalDS(values_scaled[t_train:t_val], T_IN, T_OUT), batch_size=64)
test_dl  = DataLoader(SpatioTemporalDS(values_scaled[t_val:],     T_IN, T_OUT), batch_size=64)

print(f'✅ Train: {len(train_dl.dataset):,}  Val: {len(val_dl.dataset):,}  Test: {len(test_dl.dataset):,}')

In [ ]:
# ── Model Definition ──────────────────────────────────────────────────────────
class ResidualGCN(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.gcn  = GCNConv(in_dim, out_dim)
        self.bn   = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
    def forward(self, x, ei, ew=None):
        h = TF.relu(self.bn(self.gcn(x, ei, ew)))
        return self.drop(h) + self.proj(x)


class STGNN_LSTM(nn.Module):
    """
    Architecture:
      For each timestep t:  node features → GCN layers  (captures spatial)
      Stack T outputs       → LSTM across time          (captures temporal)
      LSTM last hidden      → FC head                   (predict t+1)
    """
    def __init__(self, n_nodes, t_in, gcn_h=64, gcn_l=2, lstm_h=128, lstm_l=2, drop=0.2):
        super().__init__()
        self.n_nodes = n_nodes; self.t_in = t_in; self.gcn_h = gcn_h
        # GCN stack — shared weights across all time steps
        dims = [1] + [gcn_h] * gcn_l
        self.gcn_stack = nn.ModuleList([ResidualGCN(dims[i], dims[i+1], drop) for i in range(gcn_l)])
        # LSTM
        self.lstm = nn.LSTM(n_nodes * gcn_h, lstm_h, lstm_l,
                            batch_first=True, dropout=drop if lstm_l > 1 else 0)
        # Output head
        self.head = nn.Sequential(
            nn.Linear(lstm_h, lstm_h // 2), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(lstm_h // 2, n_nodes), nn.Sigmoid()
        )

    def forward(self, x, ei, ew=None):
        B, T, Nn = x.shape   # batch, timesteps, nodes
        gcn_seq  = []
        for t in range(T):
            h = x[:, t, :].reshape(B * Nn, 1)
            ei_b = torch.cat([ei + b * Nn for b in range(B)], dim=1)
            ew_b = ew.repeat(B) if ew is not None else None
            for gcn in self.gcn_stack:
                h = gcn(h, ei_b, ew_b)
            gcn_seq.append(h.reshape(B, Nn * self.gcn_h))
        out, _ = self.lstm(torch.stack(gcn_seq, dim=1))
        return self.head(out[:, -1, :])


model = STGNN_LSTM(N, T_IN).to(DEVICE)
print(f'✅ STGNN-LSTM  →  {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters')

---
## 🏋️ Cell 7 — Training

In [ ]:
EPOCHS   = 40
opt      = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched    = ReduceLROnPlateau(opt, factor=0.5, patience=4, verbose=True)
crit     = nn.HuberLoss(delta=0.5)
ei_dev   = edge_index.to(DEVICE)
ew_dev   = edge_weight.to(DEVICE)

history  = {'train': [], 'val': []}
best_val = float('inf')
patience = 8; no_imp = 0

def run_epoch(dl, train):
    model.train() if train else model.eval()
    total = 0
    ctx   = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train: opt.zero_grad()
            pred = model(xb, ei_dev, ew_dev)
            loss = crit(pred, yb)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            total += loss.item()
    return total / len(dl)

print(f'🚀 Training on {DEVICE} for up to {EPOCHS} epochs...\n')
t0 = time.time()

for ep in range(1, EPOCHS + 1):
    tl = run_epoch(train_dl, True)
    vl = run_epoch(val_dl,   False)
    history['train'].append(tl)
    history['val'].append(vl)
    sched.step(vl)

    flag = ''
    if vl < best_val:
        best_val = vl; no_imp = 0
        torch.save(model.state_dict(), PROCESSED / 'best_model.pt')
        flag = ' ← best'
    else:
        no_imp += 1

    if ep % 5 == 0 or ep == 1:
        print(f'  Ep {ep:3d}/{EPOCHS}  train={tl:.5f}  val={vl:.5f}{flag}')

    if no_imp >= patience:
        print(f'\n⏹  Early stop at epoch {ep}')
        break

print(f'\n✅ Training done in {(time.time()-t0)/60:.1f} min | Best val: {best_val:.5f}')

---
## 📊 Cell 8 — Evaluation + Python Visualizations

In [ ]:
# ── Run test inference ────────────────────────────────────────────────────────
model.load_state_dict(torch.load(PROCESSED / 'best_model.pt', map_location=DEVICE))
model.eval()

all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        pred = model(xb.to(DEVICE), ei_dev, ew_dev).cpu().numpy()
        all_pred.append(pred); all_true.append(yb.numpy())

preds_s   = np.concatenate(all_pred)   # [samples, N] — scaled
targets_s = np.concatenate(all_true)
preds     = scaler.inverse_transform(preds_s)    # µg/m³
targets   = scaler.inverse_transform(targets_s)

mae   = mean_absolute_error(targets.flatten(), preds.flatten())
rmse  = np.sqrt(mean_squared_error(targets.flatten(), preds.flatten()))
r2    = r2_score(targets.flatten(), preds.flatten())
mape  = np.mean(np.abs((targets - preds) / (targets + 1e-5))) * 100
bias  = np.mean(preds.flatten() - targets.flatten())

# Persistence baseline
p_mae  = mean_absolute_error(targets[1:].flatten(), targets[:-1].flatten())
p_rmse = np.sqrt(mean_squared_error(targets[1:].flatten(), targets[:-1].flatten()))

print('═'*50)
print('  Test Set Metrics — STGNN-LSTM')
print('═'*50)
print(f'  MAE    {mae:.3f}  µg/m³  (baseline: {p_mae:.3f})')
print(f'  RMSE   {rmse:.3f}  µg/m³  (baseline: {p_rmse:.3f})')
print(f'  R²     {r2:.4f}')
print(f'  MAPE   {mape:.2f}%')
print(f'  Bias   {bias:+.3f}  µg/m³')
print(f'  MAE improvement over persistence: {(p_mae-mae)/p_mae*100:.1f}%')
print('═'*50)

In [ ]:
# ── Figure 8: Training Curves ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

epochs_x = range(1, len(history['train']) + 1)
axes[0].plot(epochs_x, history['train'], label='Train', color='#2196F3', lw=2)
axes[0].plot(epochs_x, history['val'],   label='Val',   color='#F44336', lw=2)
axes[0].fill_between(epochs_x, history['train'], history['val'], alpha=0.1, color='purple')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Huber Loss')
axes[0].set_title('Training & Validation Loss Curves'); axes[0].legend()

# Learning curve smoothed
def smooth(vals, w=3):
    return pd.Series(vals).rolling(w, min_periods=1).mean().values
axes[1].plot(epochs_x, smooth(history['train']), label='Train (smoothed)', color='#2196F3', lw=2)
axes[1].plot(epochs_x, smooth(history['val']),   label='Val (smoothed)',   color='#F44336', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Huber Loss (smoothed)')
axes[1].set_title('Smoothed Learning Curves'); axes[1].legend()

plt.suptitle('STGNN-LSTM Model Training', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'fig8_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 8 saved.')

In [ ]:
# ── Figure 9: Forecast vs Actual — Full Evaluation Panel ─────────────────────
SID   = 0        # station index
NPTS  = 336      # 2 weeks
yt    = targets[:NPTS, SID]
yp    = preds[:NPTS, SID]
hrs   = np.arange(NPTS)

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.28)

# 9a: Main time series
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(hrs, yt, label='Actual',    color='#2196F3', lw=1.5, alpha=0.9)
ax1.plot(hrs, yp, label='Predicted', color='#F44336', lw=1.5, ls='--', alpha=0.9)
ax1.fill_between(hrs, yt, yp, alpha=0.15, color='purple', label='Error')
ax1.axhline(35, color='darkred', ls=':', lw=1.2, alpha=0.7, label='EPA Standard')
# Mark high-error regions
errors_ts = np.abs(yt - yp)
high_err  = errors_ts > np.percentile(errors_ts, 85)
ax1.scatter(hrs[high_err], yt[high_err], color='orange', s=15, zorder=5, label='High error hours')
ax1.set_xlabel('Hour'); ax1.set_ylabel('PM2.5 (µg/m³)')
ax1.set_title(f'1-Hour Ahead PM2.5 Forecast — Station {STATIONS[SID]}  '
              f'(MAE={mae:.2f} µg/m³, R²={r2:.3f})')
ax1.legend(fontsize=9, ncol=5); ax1.grid(alpha=0.3)

# 9b: Scatter predicted vs actual
ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(targets.flatten(), preds.flatten(), alpha=0.1, s=5, color='steelblue')
lim = max(targets.max(), preds.max()) * 1.05
ax2.plot([0, lim], [0, lim], 'r--', lw=2, label='Perfect')
ax2.set_xlabel('Actual PM2.5 (µg/m³)'); ax2.set_ylabel('Predicted PM2.5 (µg/m³)')
ax2.set_title(f'Predicted vs Actual  (R²={r2:.3f})')
ax2.legend(); ax2.grid(alpha=0.3)

# 9c: Residuals distribution
ax3 = fig.add_subplot(gs[1, 1])
resids = preds.flatten() - targets.flatten()
ax3.hist(resids, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax3.axvline(0, color='red', lw=1.5, ls='--', label='Zero bias')
ax3.axvline(bias, color='orange', lw=1.5, label=f'Mean bias={bias:+.2f}')
ax3.set_xlabel('Residual (predicted − actual, µg/m³)'); ax3.set_ylabel('Count')
ax3.set_title('Residual Distribution'); ax3.legend()

# 9d: Model vs Baseline bar chart
ax4 = fig.add_subplot(gs[2, 0])
metrics_names = ['MAE', 'RMSE']
model_vals    = [mae, rmse]
base_vals     = [p_mae, p_rmse]
x = np.arange(len(metrics_names))
w = 0.35
ax4.bar(x - w/2, model_vals, w, label='STGNN-LSTM', color='#2196F3', edgecolor='white')
ax4.bar(x + w/2, base_vals,  w, label='Persistence', color='#F44336', edgecolor='white')
ax4.set_xticks(x); ax4.set_xticklabels(metrics_names)
ax4.set_ylabel('Error (µg/m³)'); ax4.set_title('STGNN-LSTM vs Persistence Baseline')
ax4.legend()
for xi, (mv, bv) in zip(x, zip(model_vals, base_vals)):
    ax4.text(xi-w/2, mv+0.05, f'{mv:.2f}', ha='center', fontsize=9)
    ax4.text(xi+w/2, bv+0.05, f'{bv:.2f}', ha='center', fontsize=9)

# 9e: Per-station MAE map
ax5 = fig.add_subplot(gs[2, 1])
station_mae = mean_absolute_error(targets, preds, multioutput='raw_values')
sc = ax5.scatter(
    station_loc['longitude'], station_loc['latitude'],
    c=station_mae, cmap='RdYlGn_r', s=25, alpha=0.85,
    edgecolors='white', linewidths=0.2
)
plt.colorbar(sc, ax=ax5, label='MAE (µg/m³)', shrink=0.85)
ax5.set_xlim(-130, -65); ax5.set_ylim(24, 50)
ax5.set_xlabel('Longitude'); ax5.set_ylabel('Latitude')
ax5.set_title('Per-Station Forecast MAE')

plt.suptitle('STGNN-LSTM — Complete Evaluation Dashboard', fontsize=16)
plt.savefig(FIGURES / 'fig9_evaluation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 9 saved.')

In [ ]:
# ── Figure 10: Multi-Step Forecast (6h & 24h recursive) ──────────────────────
def forecast_multistep(model, seed, ei, ew, n_steps, device):
    model.eval()
    window = seed.copy(); preds_ms = []
    with torch.no_grad():
        for _ in range(n_steps):
            xin   = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)
            y_hat = model(xin, ei, ew).squeeze(0).cpu().numpy()
            preds_ms.append(y_hat)
            window = np.vstack([window[1:], y_hat])
    return np.array(preds_ms)   # [n_steps, N]

seed_window = values_scaled[-T_IN:]    # last 12 hours
fc6   = scaler.inverse_transform(forecast_multistep(model, seed_window, ei_dev, ew_dev, 6,  DEVICE))
fc24  = scaler.inverse_transform(forecast_multistep(model, seed_window, ei_dev, ew_dev, 24, DEVICE))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 6h forecast — all stations (mean ± std)
ax = axes[0]
m6 = fc6.mean(axis=1); s6 = fc6.std(axis=1)
ax.fill_between(range(1,7), m6-s6, m6+s6, alpha=0.25, color='steelblue')
ax.plot(range(1,7), m6, 'o-', color='steelblue', lw=2.5, markersize=8, label='Mean forecast')
for i in range(min(10, N)):
    ax.plot(range(1,7), fc6[:,i], '--', alpha=0.3, lw=0.8, color='steelblue')
ax.axhline(35, color='red', ls=':', lw=1.5, label='EPA Std')
ax.set_xlabel('Hours Ahead'); ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('6-Hour Ahead Recursive Forecast\n(mean ± std across all stations)')
ax.set_xticks(range(1,7)); ax.legend(); ax.grid(alpha=0.3)

# 24h forecast
ax = axes[1]
m24 = fc24.mean(axis=1); s24 = fc24.std(axis=1)
ax.fill_between(range(1,25), m24-s24, m24+s24, alpha=0.2, color='darkorange')
ax.plot(range(1,25), m24, 'o-', color='darkorange', lw=2.5, markersize=6, label='Mean forecast')
ax.axhline(35, color='red', ls=':', lw=1.5, label='EPA Std')
ax.set_xlabel('Hours Ahead'); ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('24-Hour Ahead Recursive Forecast\n(mean ± std across all stations)')
ax.set_xticks(range(1,25)); ax.set_xticklabels([f'+{h}h' for h in range(1,25)], rotation=45)
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Multi-Step PM2.5 Forecast — National Average', fontsize=15)
plt.tight_layout()
plt.savefig(FIGURES / 'fig10_multistep_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 10 saved.')

---
## 📊 Cell 9 — Export for R + Complete R Script

In [ ]:
# ── Export all data R needs ────────────────────────────────────────────────────
SAMPLE = min(8000, len(preds))

# 1. Forecast vs actual (single station)
pd.DataFrame({
    'hour':      np.arange(SAMPLE),
    'actual':    targets[:SAMPLE, SID],
    'predicted': preds[:SAMPLE, SID],
    'residual':  preds[:SAMPLE, SID] - targets[:SAMPLE, SID],
}).to_csv(R_EXPORT / 'forecast_series.csv', index=False)

# 2. Per-station evaluation
st_eval = pd.DataFrame({
    'station_id': STATIONS,
    'mae':        mean_absolute_error(targets, preds, multioutput='raw_values'),
    'rmse':       np.sqrt(mean_squared_error(targets, preds, multioutput='raw_values') if hasattr(mean_squared_error, 'multioutput') else
                          [mean_squared_error(targets[:,i], preds[:,i]) for i in range(N)]),
    'latitude':   station_loc['latitude'].values,
    'longitude':  station_loc['longitude'].values,
})
st_eval = st_eval.merge(
    pm25_pd[['station_id','state_name']].drop_duplicates(), on='station_id', how='left'
)
st_eval.to_csv(R_EXPORT / 'station_evaluation.csv', index=False)

# 3. Training history
pd.DataFrame({
    'epoch':      list(range(1, len(history['train'])+1)),
    'train_loss': history['train'],
    'val_loss':   history['val'],
}).to_csv(R_EXPORT / 'training_history.csv', index=False)

# 4. 24h multi-step forecast
fc24_df = pd.DataFrame(fc24, columns=[f'st_{i}' for i in range(N)])
fc24_df['hour_ahead'] = range(1, 25)
fc24_df['mean_pm25']  = fc24.mean(axis=1)
fc24_df['std_pm25']   = fc24.std(axis=1)
fc24_df.to_csv(R_EXPORT / 'forecast_24h.csv', index=False)

# 5. EDA summary (monthly means by state)
monthly_state = (
    pm25_pd.groupby(['state_name','month','year'])['value']
    .mean().reset_index()
)
monthly_state.to_csv(R_EXPORT / 'monthly_state_pm25.csv', index=False)

# 6. Diurnal pattern
pm25_pd.groupby('hour')['value'].agg(['mean','std','median']).reset_index()\
    .to_csv(R_EXPORT / 'diurnal_pattern.csv', index=False)

# 7. Model metrics summary
pd.DataFrame([{
    'model': 'STGNN-LSTM', 'mae': mae, 'rmse': rmse, 'r2': r2,
    'mape': mape, 'bias': bias,
    'persist_mae': p_mae, 'persist_rmse': p_rmse,
    'improvement_pct': (p_mae-mae)/p_mae*100
}]).to_csv(R_EXPORT / 'model_metrics.csv', index=False)

print('✅ R export files saved to data/r_exports/:')
for f in sorted(R_EXPORT.glob('*.csv')):
    print(f'   {f.name}')

In [ ]:
# ── Write full R visualization script ─────────────────────────────────────────
r_code = r'''
# =============================================================================
# Air Pollution Forecasting — R Visualization Suite
# Run: Rscript visualize.R   OR open in RStudio
#
# Install packages (run once):
# install.packages(c("ggplot2","dplyr","tidyr","leaflet",
#                    "viridis","plotly","htmlwidgets","patchwork",
#                    "scales","ggridges","ggcorrplot","RColorBrewer"))
# =============================================================================

library(ggplot2)
library(dplyr)
library(tidyr)
library(leaflet)
library(viridis)
library(plotly)
library(htmlwidgets)
library(patchwork)      # combine ggplots with + / /
library(scales)
library(ggridges)
library(RColorBrewer)

# Paths
DATA_DIR <- "data/r_exports"
OUT_DIR  <- "data/r_figures"
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# Load all exported files
fc       <- read.csv(file.path(DATA_DIR, "forecast_series.csv"))
stations <- read.csv(file.path(DATA_DIR, "station_evaluation.csv"))
hist_df  <- read.csv(file.path(DATA_DIR, "training_history.csv"))
fc24     <- read.csv(file.path(DATA_DIR, "forecast_24h.csv"))
monthly  <- read.csv(file.path(DATA_DIR, "monthly_state_pm25.csv"))
diurnal  <- read.csv(file.path(DATA_DIR, "diurnal_pattern.csv"))
metrics  <- read.csv(file.path(DATA_DIR, "model_metrics.csv"))

EPA_STD <- 35   # µg/m³

# ── Theme ─────────────────────────────────────────────────────────────────────
theme_air <- theme_minimal(base_size = 13) +
  theme(
    plot.title    = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "grey50", size = 11),
    panel.grid.minor = element_blank(),
    legend.position  = "bottom"
  )


# =============================================================================
# R PLOT 1: Forecast vs Actual — ggplot2
# =============================================================================
fc_week <- fc[1:336, ]   # 2-week window

p_forecast <- ggplot(fc_week, aes(x = hour)) +
  geom_ribbon(aes(ymin = pmin(actual, predicted),
                  ymax = pmax(actual, predicted)),
              fill = "#9C27B0", alpha = 0.15) +
  geom_line(aes(y = actual,    color = "Actual"),    linewidth = 0.9, alpha = 0.9) +
  geom_line(aes(y = predicted, color = "Predicted"), linewidth = 0.9,
            linetype = "dashed", alpha = 0.9) +
  geom_hline(yintercept = EPA_STD, color = "#D32F2F", linetype = "dotted",
             linewidth = 0.9) +
  annotate("text", x = 20, y = EPA_STD + 1.5,
           label = "EPA 24h Standard (35 µg/m³)",
           color = "#D32F2F", size = 3.5) +
  scale_color_manual(
    values = c("Actual" = "#1565C0", "Predicted" = "#E53935"),
    name   = NULL
  ) +
  labs(
    title    = "1-Hour Ahead PM2.5 Forecast — STGNN-LSTM",
    subtitle = sprintf("MAE = %.3f µg/m³  |  R² = %.3f  |  MAPE = %.1f%%",
                       metrics$mae, metrics$r2, metrics$mape),
    x = "Hour Index",
    y = expression(PM[2.5] ~ (µg/m^3))
  ) +
  theme_air

ggsave(file.path(OUT_DIR, "R1_forecast_vs_actual.png"),
       p_forecast, width = 14, height = 5, dpi = 200)
cat("✅ R Plot 1 saved\n")


# =============================================================================
# R PLOT 2: Training Loss Curves — ggplot2
# =============================================================================
hist_long <- hist_df %>%
  pivot_longer(c(train_loss, val_loss), names_to = "split", values_to = "loss") %>%
  mutate(split = recode(split, train_loss = "Train", val_loss = "Validation"))

p_loss <- ggplot(hist_long, aes(x = epoch, y = loss, color = split)) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 1.5, alpha = 0.6) +
  geom_smooth(se = FALSE, method = "loess", linewidth = 0.6, linetype = "dashed", alpha = 0.5) +
  scale_color_manual(values = c("Train" = "#1565C0", "Validation" = "#E53935"),
                     name = NULL) +
  labs(
    title    = "STGNN-LSTM Training Curves",
    subtitle = "Huber Loss (delta=0.5) with early stopping",
    x = "Epoch", y = "Huber Loss"
  ) +
  theme_air

ggsave(file.path(OUT_DIR, "R2_training_curves.png"),
       p_loss, width = 10, height = 5, dpi = 200)
cat("✅ R Plot 2 saved\n")


# =============================================================================
# R PLOT 3: Residuals Analysis — 4-panel
# =============================================================================
# 3a: Residual histogram
p3a <- ggplot(fc, aes(x = residual)) +
  geom_histogram(bins = 60, fill = "#1565C0", color = "white", alpha = 0.8) +
  geom_vline(xintercept = 0,           color = "#D32F2F", linewidth = 1.2, linetype = "dashed") +
  geom_vline(xintercept = mean(fc$residual), color = "#FF6F00", linewidth = 1,
             linetype = "dashed") +
  labs(title = "Residual Distribution",
       x = "Residual (predicted − actual, µg/m³)", y = "Count") +
  theme_air

# 3b: Residual vs actual (heteroscedasticity check)
p3b <- ggplot(fc, aes(x = actual, y = residual)) +
  geom_hex(bins = 50, aes(fill = after_stat(count))) +
  scale_fill_viridis(option = "plasma", name = "Count") +
  geom_hline(yintercept = 0, color = "red", linewidth = 1, linetype = "dashed") +
  geom_smooth(method = "loess", se = TRUE, color = "white", linewidth = 0.8) +
  labs(title = "Residuals vs Actual",
       x = expression(Actual ~ PM[2.5]), y = "Residual") +
  theme_air + theme(legend.position = "right")

# 3c: Predicted vs Actual hex
p3c <- ggplot(fc, aes(x = actual, y = predicted)) +
  geom_hex(bins = 60, aes(fill = after_stat(count))) +
  scale_fill_viridis(option = "magma", name = "Count") +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1.2, linetype = "dashed") +
  labs(title = expression(bold("Predicted vs Actual PM"[2.5])),
       x = expression(Actual ~ (µg/m^3)),
       y = expression(Predicted ~ (µg/m^3))) +
  theme_air + theme(legend.position = "right")

# 3d: Model vs baseline comparison
comp_df <- data.frame(
  model  = c("STGNN-LSTM", "STGNN-LSTM", "Persistence", "Persistence"),
  metric = c("MAE", "RMSE", "MAE", "RMSE"),
  value  = c(metrics$mae, metrics$rmse, metrics$persist_mae, metrics$persist_rmse)
)
p3d <- ggplot(comp_df, aes(x = metric, y = value, fill = model)) +
  geom_col(position = "dodge", width = 0.6, color = "white") +
  geom_text(aes(label = round(value, 2)), position = position_dodge(0.6),
            vjust = -0.4, size = 3.5, fontface = "bold") +
  scale_fill_manual(values = c("STGNN-LSTM" = "#1565C0", "Persistence" = "#E53935"),
                    name = NULL) +
  labs(title = "Model vs Persistence Baseline",
       subtitle = sprintf("MAE improvement: %.1f%%", metrics$improvement_pct),
       x = "Metric", y = expression(Error ~ (µg/m^3))) +
  theme_air

p_residuals <- (p3a | p3b) / (p3c | p3d) +
  plot_annotation(title = "STGNN-LSTM — Error & Residual Analysis",
                  theme = theme(plot.title = element_text(face="bold", size=16)))

ggsave(file.path(OUT_DIR, "R3_residual_analysis.png"),
       p_residuals, width = 16, height = 12, dpi = 200)
cat("✅ R Plot 3 saved\n")


# =============================================================================
# R PLOT 4: 24-Hour Ahead Forecast with Uncertainty Band
# =============================================================================
p_fc24 <- ggplot(fc24, aes(x = hour_ahead)) +
  geom_ribbon(aes(ymin = mean_pm25 - std_pm25,
                  ymax = mean_pm25 + std_pm25),
              fill = "#FF6F00", alpha = 0.25) +
  geom_ribbon(aes(ymin = mean_pm25 - 0.5*std_pm25,
                  ymax = mean_pm25 + 0.5*std_pm25),
              fill = "#FF6F00", alpha = 0.35) +
  geom_line(aes(y = mean_pm25), color = "#E65100", linewidth = 1.8) +
  geom_point(aes(y = mean_pm25), color = "#E65100", size = 3) +
  geom_hline(yintercept = EPA_STD, color = "#D32F2F", linewidth = 1,
             linetype = "dashed") +
  annotate("text", x = 2, y = EPA_STD + 1,
           label = "EPA Standard (35 µg/m³)",
           color = "#D32F2F", size = 3.5, hjust = 0) +
  scale_x_continuous(breaks = seq(1, 24, 2),
                     labels = paste0("+", seq(1, 24, 2), "h")) +
  labs(
    title    = "24-Hour Ahead PM2.5 Forecast — Recursive Prediction",
    subtitle = "National average across all stations  |  Shaded bands: ±0.5σ and ±1σ",
    x        = "Hours Ahead",
    y        = expression(PM[2.5] ~ (µg/m^3))
  ) +
  theme_air

ggsave(file.path(OUT_DIR, "R4_24h_forecast.png"),
       p_fc24, width = 13, height = 6, dpi = 200)
cat("✅ R Plot 4 saved\n")


# =============================================================================
# R PLOT 5: Seasonal + Diurnal Patterns
# =============================================================================
month_labels <- c("Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec")

# 5a: Monthly PM2.5 ridge plot by state (top 15 states by data volume)
top_states <- monthly %>%
  count(state_name, sort = TRUE) %>%
  head(12) %>%
  pull(state_name)

p5a <- monthly %>%
  filter(state_name %in% top_states) %>%
  mutate(month_f = factor(month, levels = 1:12, labels = month_labels)) %>%
  ggplot(aes(x = value, y = reorder(state_name, value, median),
             fill = after_stat(x))) +
  ggridges::geom_density_ridges_gradient(scale = 1.5, rel_min_height = 0.01) +
  scale_fill_viridis(option = "inferno", name = "PM2.5") +
  geom_vline(xintercept = EPA_STD, color = "white", linewidth = 0.8, linetype = "dashed") +
  labs(title = "PM2.5 Distribution by State — Ridge Plot",
       subtitle = "Top 12 states by data volume | White dashed = EPA standard",
       x = expression(PM[2.5] ~ (µg/m^3)), y = "State") +
  theme_air + theme(legend.position = "right")

# 5b: Diurnal pattern
p5b <- ggplot(diurnal, aes(x = hour)) +
  geom_ribbon(aes(ymin = mean - std, ymax = mean + std),
              fill = "#1565C0", alpha = 0.2) +
  geom_line(aes(y = mean),   color = "#1565C0", linewidth = 1.8) +
  geom_line(aes(y = median), color = "#FF6F00", linewidth = 1.2, linetype = "dashed") +
  geom_point(aes(y = mean),  color = "#1565C0", size = 2.5) +
  annotate("rect", xmin = 6, xmax = 9, ymin = -Inf, ymax = Inf,
           fill = "yellow", alpha = 0.15) +
  annotate("rect", xmin = 17, xmax = 20, ymin = -Inf, ymax = Inf,
           fill = "orange", alpha = 0.12) +
  annotate("text", x = 7.5, y = Inf, vjust = 1.5, size = 3,
           label = "Morning\nRush") +
  annotate("text", x = 18.5, y = Inf, vjust = 1.5, size = 3,
           label = "Evening\nRush") +
  scale_x_continuous(breaks = seq(0,23,3),
                     labels = paste0(seq(0,23,3), ":00")) +
  labs(title = "Diurnal PM2.5 Pattern",
       subtitle = "Mean (blue) and median (orange dashed)  |  Shaded: ±1σ",
       x = "Hour of Day",
       y = expression(PM[2.5] ~ (µg/m^3))) +
  theme_air

p_patterns <- p5a / p5b +
  plot_annotation(title = "Seasonal & Diurnal PM2.5 Patterns — 2022–2023",
                  theme = theme(plot.title = element_text(face="bold", size=16)))

ggsave(file.path(OUT_DIR, "R5_seasonal_diurnal.png"),
       p_patterns, width = 14, height = 13, dpi = 200)
cat("✅ R Plot 5 saved\n")


# =============================================================================
# R PLOT 6: State-Level Bar Chart (Top 20 worst states)
# =============================================================================
state_summary <- monthly %>%
  group_by(state_name) %>%
  summarise(mean_pm25 = mean(value, na.rm=TRUE),
            sd_pm25   = sd(value, na.rm=TRUE)) %>%
  arrange(desc(mean_pm25)) %>%
  head(20)

p_states <- ggplot(state_summary,
                   aes(x = reorder(state_name, mean_pm25), y = mean_pm25,
                       fill = mean_pm25)) +
  geom_col(color = "white", linewidth = 0.4) +
  geom_errorbar(aes(ymin = mean_pm25 - sd_pm25/2,
                    ymax = mean_pm25 + sd_pm25/2),
                width = 0.4, color = "grey40") +
  geom_hline(yintercept = 12, color = "#FF6F00", linewidth = 1,
             linetype = "dashed") +
  geom_hline(yintercept = 9, color = "#388E3C", linewidth = 1,
             linetype = "dashed") +
  annotate("text", x = 1.5, y = 12.5, label = "NAAQS Annual (12 µg/m³)",
           color = "#FF6F00", size = 3, hjust = 0) +
  annotate("text", x = 1.5, y = 9.5, label = "WHO Guideline (9 µg/m³)",
           color = "#388E3C", size = 3, hjust = 0) +
  scale_fill_gradient2(low = "#388E3C", mid = "#FFC107", high = "#D32F2F",
                       midpoint = 10, guide = "none") +
  coord_flip() +
  labs(title = "Top 20 States — Mean Annual PM2.5 (2022–2023)",
       subtitle = "Error bars = ±0.5 SD  |  Horizontal lines = regulatory standards",
       x = NULL, y = expression(Mean ~ PM[2.5] ~ (µg/m^3))) +
  theme_air

ggsave(file.path(OUT_DIR, "R6_state_pm25.png"),
       p_states, width = 11, height = 8, dpi = 200)
cat("✅ R Plot 6 saved\n")


# =============================================================================
# R PLOT 7: Interactive Leaflet Map — Station Forecast Error
# =============================================================================
stations_clean <- stations %>%
  filter(!is.na(latitude), !is.na(longitude), !is.na(mae),
         longitude > -130, longitude < -65,
         latitude  >   24, latitude  <  50)

pal_mae <- colorNumeric(
  palette = c("#388E3C", "#FFC107", "#D32F2F"),
  domain   = stations_clean$mae,
  na.color = "#BDBDBD"
)

map_mae <- leaflet(stations_clean) %>%
  addProviderTiles(providers$CartoDB.Positron) %>%
  addCircleMarkers(
    lng         = ~longitude,
    lat         = ~latitude,
    radius      = ~scales::rescale(mae, to = c(4, 12)),
    color       = ~pal_mae(mae),
    stroke      = TRUE, weight = 1.2, opacity = 0.9,
    fillOpacity = 0.85,
    popup = ~paste0(
      "<b>Station:</b> ", station_id, "<br>",
      "<b>State:</b> ",   state_name,  "<br>",
      "<b>MAE:</b> ",     round(mae, 3), " µg/m³<br>",
      "<b>RMSE:</b> ",    round(rmse, 3), " µg/m³"
    ),
    label = ~paste0(station_id, " | MAE=" , round(mae,2))
  ) %>%
  addLegend(
    position = "bottomright",
    pal    = pal_mae, values = ~mae,
    title  = "Forecast MAE (µg/m³)",
    opacity = 0.9
  ) %>%
  addControl(
    html = paste0(
      "<div style='background:white;padding:8px;border-radius:6px;font-size:13px;'>",
      "<b>STGNN-LSTM Forecast Error</b><br>",
      "National MAE: ", round(mean(stations_clean$mae, na.rm=TRUE), 3), " µg/m³",
      "</div>"
    ),
    position = "topright"
  )

saveWidget(map_mae,
           file.path(OUT_DIR, "R7_station_mae_map.html"),
           selfcontained = TRUE)
cat("✅ R Plot 7 (interactive map) saved\n")


# =============================================================================
# R PLOT 8: Interactive Plotly — Monthly PM2.5 Trend
# =============================================================================
monthly_nat <- monthly %>%
  mutate(date = as.Date(paste(year, month, "01", sep="-"))) %>%
  group_by(date) %>%
  summarise(mean_pm25 = mean(value, na.rm=TRUE),
            sd_pm25   = sd(value, na.rm=TRUE))

p_trend <- plot_ly(monthly_nat) %>%
  add_ribbons(
    x = ~date,
    ymin = ~mean_pm25 - sd_pm25,
    ymax = ~mean_pm25 + sd_pm25,
    fillcolor = "rgba(33,150,243,0.2)",
    line = list(color = "transparent"),
    name = "±1 SD"
  ) %>%
  add_lines(
    x = ~date, y = ~mean_pm25,
    line = list(color = "#1565C0", width = 2.5),
    name = "National Mean PM2.5"
  ) %>%
  add_lines(
    x = range(monthly_nat$date),
    y = c(EPA_STD, EPA_STD),
    line = list(color = "red", dash = "dot", width = 1.5),
    name = "EPA Standard"
  ) %>%
  layout(
    title  = list(text = "National Monthly PM2.5 Trend 2022–2023", font = list(size=16)),
    xaxis  = list(title = "Date"),
    yaxis  = list(title = "PM2.5 (µg/m³)"),
    legend = list(orientation = "h", x = 0, y = -0.15),
    hovermode = "x unified"
  )

saveWidget(p_trend,
           file.path(OUT_DIR, "R8_monthly_trend_plotly.html"),
           selfcontained = TRUE)
cat("✅ R Plot 8 (interactive Plotly trend) saved\n")


# =============================================================================
# Summary
# =============================================================================
cat("\n", strrep("=", 55), "\n")
cat("  ✅  ALL R VISUALIZATIONS SAVED\n")
cat(strrep("=", 55), "\n")
cat("  R1  Forecast vs Actual          → R1_forecast_vs_actual.png\n")
cat("  R2  Training Loss Curves        → R2_training_curves.png\n")
cat("  R3  Residual Analysis (4-panel) → R3_residual_analysis.png\n")
cat("  R4  24h Forecast + Uncertainty  → R4_24h_forecast.png\n")
cat("  R5  Seasonal & Diurnal Patterns → R5_seasonal_diurnal.png\n")
cat("  R6  State-Level PM2.5 Bar Chart → R6_state_pm25.png\n")
cat("  R7  Interactive Leaflet Map     → R7_station_mae_map.html\n")
cat("  R8  Interactive Plotly Trend    → R8_monthly_trend_plotly.html\n")
cat(strrep("=", 55), "\n")
'''

r_script_path = Path('visualize.R')
r_script_path.write_text(r_code)
print(f'✅ R script written → {r_script_path}')
print('   Run with:  Rscript visualize.R')
print('   Or open in RStudio and press "Source"')

---
## 💾 Cell 10 — Save Everything & Summary

In [ ]:
# ── Save full checkpoint ──────────────────────────────────────────────────────
torch.save({
    'model_state':  model.state_dict(),
    'model_config': dict(n_nodes=N, t_in=T_IN),
    'stations':     STATIONS,
    'edge_index':   edge_index,
    'edge_weight':  edge_weight,
    'scaler':       scaler,
    'metrics':      dict(mae=mae, rmse=rmse, r2=r2, mape=mape, bias=bias),
    'history':      history,
}, PROCESSED / 'stgnn_lstm_checkpoint.pt')

spark.stop()

print()
print('═'*58)
print('  ✅  PIPELINE COMPLETE')
print('═'*58)
print()
print('📁 Saved artifacts:')
print('   data/processed/pm25_engineered.parquet  — Spark output')
print('   data/processed/station_metadata.csv     — station info')
print('   data/processed/stgnn_lstm_checkpoint.pt — model')
print()
print('📊 Python figures (data/figures/):')
for f in sorted(FIGURES.glob('*.png')):
    print(f'   {f.name}')
print()
print('📊 R exports (data/r_exports/):')
for f in sorted(R_EXPORT.glob('*.csv')):
    print(f'   {f.name}')
print()
print('📊 R script:  visualize.R')
print('   → Run in RStudio to generate 8 publication-quality plots')
print()
print(f'🏆 Final Test MAE:  {mae:.3f} µg/m³')
print(f'   Test R²:         {r2:.4f}')
print(f'   vs Persistence:  {(p_mae-mae)/p_mae*100:.1f}% better')
print('═'*58)

---
## 📋 Quick Reference

| What | Where | How to run |
|---|---|---|
| This notebook | `air_pollution_forecasting.ipynb` | Jupyter |
| R plots | `visualize.R` | `Rscript visualize.R` or RStudio |
| Model checkpoint | `data/processed/stgnn_lstm_checkpoint.pt` | `torch.load(...)` |
| Spark output | `data/processed/pm25_engineered.parquet` | `spark.read.parquet(...)` |
| R data exports | `data/r_exports/*.csv` | `read.csv(...)` |

### R Visualizations produced
| File | Content |
|---|---|
| `R1_forecast_vs_actual.png` | Forecast vs actual time series with EPA standard line |
| `R2_training_curves.png` | Train/val loss with smoothed trend |
| `R3_residual_analysis.png` | 4-panel: histogram + hex scatter + residuals + model vs baseline |
| `R4_24h_forecast.png` | 24h ahead forecast with ±0.5σ and ±1σ uncertainty bands |
| `R5_seasonal_diurnal.png` | Ridge plot by state + diurnal pattern with rush-hour annotations |
| `R6_state_pm25.png` | State bar chart with NAAQS and WHO guidelines |
| `R7_station_mae_map.html` | Interactive leaflet map — station forecast error |
| `R8_monthly_trend_plotly.html` | Interactive Plotly — national monthly PM2.5 trend |